In [1]:
import pandas as pd
import numpy as np
import optuna
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

In [2]:
# ---------------------------------------------------------
# 1. Load the Datasets & Filter for Classes 2, 3, 4
# ---------------------------------------------------------
print("Loading datasets...")

DATA_PATH = '../processed_datasets/'
X_TRAIN_PATH = DATA_PATH + 'X_train_catboost.parquet'
X_VAL_PATH = DATA_PATH + 'X_val_catboost.parquet'
X_TEST_PATH = DATA_PATH + 'X_test_catboost.parquet'

Y_TRAIN_PATH = DATA_PATH + 'y_train.csv'
Y_VAL_PATH = DATA_PATH + 'y_val.csv'
Y_TEST_PATH = DATA_PATH + 'y_test.csv'

X_train = pd.read_parquet(X_TRAIN_PATH, engine="fastparquet")
X_val = pd.read_parquet(X_VAL_PATH, engine="fastparquet")
X_test = pd.read_parquet(X_TEST_PATH, engine="fastparquet")

y_train = pd.read_csv(Y_TRAIN_PATH).squeeze()
y_val = pd.read_csv(Y_VAL_PATH).squeeze()
y_test = pd.read_csv(Y_TEST_PATH).squeeze()

# EXCLUSIVE FILTER: Only keep observations for classes 2, 3, and 4
mask_train = y_train >= 2
mask_val = y_val >= 2
mask_test = y_test >= 2

X_train, y_train = X_train[mask_train], y_train[mask_train]
X_val, y_val = X_val[mask_val], y_val[mask_val]
X_test, y_test = X_test[mask_test], y_test[mask_test]

print(f"Filtered for Classes 2, 3, 4.")
print(f"Train shapes: X={X_train.shape}, y={y_train.shape}")
print(f"Validation shapes: X={X_val.shape}, y={y_val.shape}")
print(f"Test shapes: X={X_test.shape}, y={y_test.shape}\n")

categorical_features = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
categorical_features.extend(['DAY_OF_WEEK'])
print(f"Detected {len(categorical_features)} categorical features.")

Loading datasets...
Filtered for Classes 2, 3, 4.
Train shapes: X=(189090, 32), y=(189090,)
Validation shapes: X=(15296, 32), y=(15296,)
Test shapes: X=(11189, 32), y=(11189,)

Detected 11 categorical features.


In [3]:
# ---------------------------------------------------------
# 2. Define the Optuna Objective Function
# ---------------------------------------------------------
static_params = {
    "random_seed": 42,
    "verbose": False,
    "task_type": "GPU",
    "auto_class_weights": "Balanced",
    "eval_metric": "TotalF1:average=Macro"
}

def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 500, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 50),
    }
    params.update(static_params)
    
    model = CatBoostClassifier(**params)

    model.fit(
        X_train, 
        y_train,
        cat_features=categorical_features,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=100, 
        use_best_model=True
    )

    preds_class = np.squeeze(model.predict(X_val))
    macro_f1 = f1_score(y_val, preds_class, average='macro')
    
    return macro_f1

In [4]:
# ---------------------------------------------------------
# 3. Run the Optuna Study
# ---------------------------------------------------------
print("Starting Optuna hyperparameter tuning for Classes 2-4...")

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30) 

print("\n--- Optimization Finished ---")
print(f"Best Macro F1: {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-06-09 17:28:22,699] A new study created in memory with name: no-name-21be8292-47c4-4248-965a-66ee4c104f65


Starting Optuna hyperparameter tuning for Classes 2-4...


[I 2026-06-09 17:28:25,542] Trial 0 finished with value: 0.4765507707721091 and parameters: {'iterations': 1104, 'learning_rate': 0.0011024510378080904, 'depth': 4, 'l2_leaf_reg': 3.8806533623032995, 'min_data_in_leaf': 46}. Best is trial 0 with value: 0.4765507707721091.
[I 2026-06-09 17:28:31,521] Trial 1 finished with value: 0.5446784698279314 and parameters: {'iterations': 1804, 'learning_rate': 0.04219598229272793, 'depth': 8, 'l2_leaf_reg': 9.476756835143602, 'min_data_in_leaf': 8}. Best is trial 1 with value: 0.5446784698279314.
[I 2026-06-09 17:28:36,556] Trial 2 finished with value: 0.5407554077297455 and parameters: {'iterations': 631, 'learning_rate': 0.03048185901044047, 'depth': 4, 'l2_leaf_reg': 7.6904856575930856, 'min_data_in_leaf': 24}. Best is trial 1 with value: 0.5446784698279314.
[I 2026-06-09 17:28:59,083] Trial 3 finished with value: 0.5520121444791241 and parameters: {'iterations': 1486, 'learning_rate': 0.016158385201186305, 'depth': 10, 'l2_leaf_reg': 2.773393


--- Optimization Finished ---
Best Macro F1: 0.5553
Best Parameters:
  iterations: 1334
  learning_rate: 0.0451814165870168
  depth: 10
  l2_leaf_reg: 2.944686827124794
  min_data_in_leaf: 11


In [7]:
# ---------------------------------------------------------
# 4. Train Final Model & Evaluate on Test Set
# ---------------------------------------------------------
print("\nTraining final model for Classes 2-4...")

final_params = study.best_params.copy()
final_params.update(static_params)
final_model = CatBoostClassifier(**final_params)

final_model.fit(
    X_train, 
    y_train,
    cat_features=categorical_features,
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=100,
    use_best_model=True
)

y_pred = final_model.predict(X_test)
print(f"Final Test Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))


Training final model for Classes 2-4...
Final Test Accuracy: 0.6692

Classification Report:
              precision    recall  f1-score   support

           2       0.84      0.76      0.80      7736
           3       0.34      0.37      0.36      2274
           4       0.45      0.64      0.52      1179

    accuracy                           0.67     11189
   macro avg       0.54      0.59      0.56     11189
weighted avg       0.70      0.67      0.68     11189



In [8]:
final_model.save_model("catboost_model_2to4.cbm")